In [42]:
# Local path config

import sys
from pathlib import Path

current_dir = Path.cwd()

root_dir = current_dir
while not (root_dir / 'utils').is_dir() and root_dir != root_dir.parent:
    root_dir = root_dir.parent

if str(root_dir) not in sys.path:
    sys.path.insert(0, str(root_dir))

# Decision Tree Regressor - From scratch and `scikit-learn` approach

### 1. Cấu trúc Node
* **Decision Node:** Lưu `feature` và `threshold` để chia dữ liệu, kèm con trỏ `left`, `right`. Thuộc tính `value = None`.
* **Leaf Node:** Chỉ lưu `value` (giá trị dự đoán). Các thuộc tính `feature`, `threshold`, `left`, `right` đều bằng `None`.

### 2. Các bước thực hiện

* **Bước 1 (Khởi tạo):** Bắt đầu tại Root Node với toàn bộ tập dữ liệu `Training`.
* **Bước 2 (Chọn feature và threshold):** 
  * Duyệt qua các feature và các ngưỡng giá trị (ngưỡng thường là trung bình của các giá trị liên tiếp sau khi sắp xếp).
  * Thử chia dữ liệu và tính tổng MAE của 2 nhánh. Cặp `(feature, threshold)` nào làm **tối thiểu hóa tổng MAE** sẽ được chọn.
* **Bước 3 (Chia node con):** Chia tập dữ liệu hiện tại thành 2 tập con vật lý:
  * **Nhánh bên trái (`left`):** Chứa các dữ liệu có giá trị feature $\le \text{threshold}$.
  * **Nhánh bên phải (`right`):** Chứa các dữ liệu có giá trị feature $> \text{threshold}$.
* **Bước 4 (Đệ quy):** Lặp lại Bước 2 và Bước 3 cho các node `left` và `right` cho đến khi đạt điều kiện dừng (ví dụ: đạt `max_depth`, `samples` khi chia quá ít).
* **Bước 5 (Gán output tại Leaf):** Khi dừng, biến node đó thành Node lá. Gán `value` bằng **Median** (hoặc **Mean**) của tập dữ liệu tại node lá đó; gán `left = None` và `right = None`.

## Triển khai `DTR` không dùng thư viện

Import thư viện cần thiết trước khi code

In [69]:
from __future__ import annotations

import pandas as pd 
import numpy as np
import joblib
import sklearn.tree
from dataclasses import dataclass
from practice_2.utils.custom_hyperparameter_tuning import CustomGridSearchCV
from practice_2.utils.custom_cv import CustomKFold

X_train = joblib.load('./data/ready_for_train/X_train_final.pkl')
X_test = joblib.load('./data/ready_for_train/X_test_final.pkl')

y_train = joblib.load('./data/ready_for_train/y_train_log.pkl')
y_test = joblib.load('./data/ready_for_train/y_test_log.pkl')

# Previewing Train, test shapes

print('Training dataset shape:')
print(f'X: {X_train.shape}')
print(f'y: {y_train.shape}')

print('Testing dataset shape:')
print(f'X: {X_test.shape}')
print(f'y: {y_test.shape}')


Training dataset shape:
X: (79448, 31)
y: (79448,)
Testing dataset shape:
X: (19862, 31)
y: (19862,)


Triển khai Decision Tree Regressor không dùng `sklearn`

In [ ]:
@dataclass
class Node:
    feature: int | None = None
    threshold: float | None = None
    left: "Node | None" = None
    right: "Node | None" = None
    value: float | None = None

    def is_leaf_node(self) -> bool:
        return self.value is not None


class DTR:
    def __init__(self, max_depth: int = 5, min_samples_split: int = 3, min_samples_leaf: int = 2):
        self.max_depth = max_depth
        self.min_samples_split = min_samples_split
        self.min_samples_leaf = min_samples_leaf
        self.root: Node | None = None

    def __mse(self, y_actual: np.ndarray, y_hat: float) -> float:
        """Tính Mean Squared Error giữa y thực tế và một giá trị dự đoán.

        Args:
            y_actual (np.ndarray): Mảng target thực tế.
            y_hat (float): Giá trị dự đoán đại diện cho node.

        Returns:
            float: Giá trị MSE.
        """
        residual = y_actual - y_hat
        return float(np.mean(residual ** 2))

    def set_params(self, **params) -> "DTR":
        for key, value in params.items():
            setattr(self, key, value)
        return self

    def get_params(self) -> dict:
        return {
            "max_depth": self.max_depth,
            "min_samples_split": self.min_samples_split,
            "min_samples_leaf": self.min_samples_leaf,
        }

    def fit(self, X: np.ndarray, y: np.ndarray) -> "DTR":
        """Train model Decision Tree Regressor với tập input là dữ liệu dạng NumPy.

        Args:
            X (np.ndarray): Ma trận feature có shape `(n_samples, n_features)`.
            y (np.ndarray): Vector target có shape `(n_samples,)`.

        Returns:
            DTR: Mô hình sau khi đã xây cây.
        """
        X = np.asarray(X)
        y = np.asarray(y).ravel()
        self.root = self.__build_tree(X, y, 0)
        return self

    def predict(self, X: np.ndarray) -> np.ndarray:
        """Dự đoán target cho dữ liệu đầu vào dạng NumPy.

        Args:
            X (np.ndarray): Ma trận feature có shape `(n_samples, n_features)`.

        Returns:
            np.ndarray: Vector dự đoán có shape `(n_samples,)`.
        """
        X = np.asarray(X)
        return np.array([self.__traverse_tree(row, self.root) for row in X])

    def __traverse_tree(self, x: np.ndarray, node: Node) -> float:
        if node.is_leaf_node():
            return node.value

        if x[node.feature] <= node.threshold:
            return self.__traverse_tree(x, node.left)

        return self.__traverse_tree(x, node.right)

    def __build_tree(self, X: np.ndarray, y: np.ndarray, depth: int) -> Node:
        n_samples = X.shape[0]

        if (depth >= self.max_depth
            or n_samples < self.min_samples_split
            or n_samples < 2 * self.min_samples_leaf
            or len(np.unique(y)) == 1):
            return Node(value=float(np.mean(y)))

        best_feature, best_threshold = self._get_best_split_criteria(X, y)

        if best_feature is None:
            return Node(value=float(np.mean(y)))

        left_mask = X[:, best_feature] <= best_threshold

        X_left = X[left_mask]
        y_left = y[left_mask]

        X_right = X[~left_mask]
        y_right = y[~left_mask]

        left_child = self.__build_tree(X_left, y_left, depth + 1)
        right_child = self.__build_tree(X_right, y_right, depth + 1)

        return Node(
            feature=best_feature,
            threshold=best_threshold,
            left=left_child,
            right=right_child,
            value=None,
        )

    def _get_best_split_criteria(self, X: np.ndarray, y: np.ndarray) -> tuple[int, float] | tuple[None, None]:
        """Tìm feature index và threshold tốt nhất để chia node hiện tại.
        (Feature index tức là ta sẽ thu thập index của best feature đó)
        Args:
            X (np.ndarray): Ma trận feature của node hiện tại, shape
                `(n_samples, n_features)`.
            y (np.ndarray): Vector target của node hiện tại, shape `(n_samples,)`.

        Returns:
            tuple[int, float] | tuple[None, None]: Cặp `(feature_index, threshold)`
            tốt nhất. Nếu không tìm được split hợp lệ, trả về `(None, None)`.

        Thuật toán chính:
            1. Tính MSE của node hiện tại làm mốc lỗi ban đầu.
            2. Duyệt qua từng feature index trong X.
            3. Với mỗi feature, tạo candidate threshold bằng midpoint giữa các
               giá trị unique liên tiếp sau khi sắp xếp.
            4. Với mỗi threshold, chia y thành nhánh trái/phải bằng boolean mask.
            5. Bỏ qua split nếu một nhánh có ít mẫu hơn `min_samples_leaf`.
            6. Tính weighted MSE của split và cập nhật split tốt nhất nếu lỗi nhỏ hơn.
        """
        current_mse = self.__mse(y, float(np.mean(y)))
        best_feature = None
        best_threshold = None
        n_features = X.shape[1]

        for feature in range(n_features):
            x_arr = X[:, feature]
            unique_vals = np.sort(np.unique(x_arr))
            splits = (unique_vals[:-1] + unique_vals[1:]) / 2.0

            for split in splits:
                left_mask = x_arr <= split

                n_left = np.sum(left_mask)
                n_right = len(y) - n_left

                if n_left < self.min_samples_leaf or n_right < self.min_samples_leaf:
                    continue

                y_left = y[left_mask]
                y_right = y[~left_mask]

                left_mse = self.__mse(y_left, float(np.mean(y_left)))
                right_mse = self.__mse(y_right, float(np.mean(y_right)))
                weighted_mse = ((left_mse * n_left) + (right_mse * n_right)) / len(y)

                if weighted_mse < current_mse:
                    current_mse = weighted_mse
                    best_feature = feature
                    best_threshold = float(split)

        return (best_feature, best_threshold)


## Dùng thư viện `sklearn-learn` để triển khai `DTR`

In [71]:
sklearn_dtr_model = sklearn.tree.DecisionTreeRegressor(max_depth = 3, min_samples_split=100, min_samples_leaf=500)

sklearn_dtr_model.fit(X_train, y_train)


,"max_depth max_depth: int, default=NoneThe maximum depth of the tree. If None, then nodes are expanded untilall leaves are pure or until all leaves contain less thanmin_samples_split samples.For an example of how ``max_depth`` influences the model, see:ref:`sphx_glr_auto_examples_tree_plot_tree_regression.py`.",3
,"min_samples_split min_samples_split: int or float, default=2The minimum number of samples required to split an internal node:- If int, then consider `min_samples_split` as the minimum number.- If float, then `min_samples_split` is a fraction and `ceil(min_samples_split * n_samples)` are the minimum number of samples for each split... versionchanged:: 0.18 Added float values for fractions.",100
,"min_samples_leaf min_samples_leaf: int or float, default=1The minimum number of samples required to be at a leaf node.A split point at any depth will only be considered if it leaves atleast ``min_samples_leaf`` training samples in each of the left andright branches. This may have the effect of smoothing the model,especially in regression.- If int, then consider `min_samples_leaf` as the minimum number.- If float, then `min_samples_leaf` is a fraction and `ceil(min_samples_leaf * n_samples)` are the minimum number of samples for each node... versionchanged:: 0.18 Added float values for fractions.",500
,"criterion criterion: {""squared_error"", ""absolute_error"", ""poisson""}, default=""squared_error""The function to measure the quality of a split. Supported criteriaare ""squared_error"" for the mean squared error, which is equal tovariance reduction as feature selection criterion and minimizes the L2loss using the mean of each terminal node, ""absolute_error"" for the meanabsolute error, which minimizes the L1 loss using the median of each terminalnode, and ""poisson"" which uses reduction in Poisson deviance to find splits,also using the mean of each terminal node... versionadded:: 0.18 Mean Absolute Error (MAE) criterion... versionadded:: 0.24 Poisson deviance criterion... versionchanged:: 1.9 Criterion `""friedman_mse""` was deprecated.",'squared_error'
,"splitter splitter: {""best"", ""random""}, default=""best""The strategy used to choose the split at each node. Supportedstrategies are ""best"" to choose the best split and ""random"" to choosethe best random split.",'best'
,"min_weight_fraction_leaf min_weight_fraction_leaf: float, default=0.0The minimum weighted fraction of the sum total of weights (of allthe input samples) required to be at a leaf node. Samples haveequal weight when sample_weight is not provided.",0.0
,"max_features max_features: int, float or {""sqrt"", ""log2""}, default=NoneThe number of features to consider when looking for the best split:- If int, then consider `max_features` features at each split.- If float, then `max_features` is a fraction and `max(1, int(max_features * n_features_in_))` features are considered at each split.- If ""sqrt"", then `max_features=sqrt(n_features)`.- If ""log2"", then `max_features=log2(n_features)`.- If None, then `max_features=n_features`.Note: the search for a split does not stop until at least onevalid partition of the node samples is found, even if it requires toeffectively inspect more than ``max_features`` features.",None
,"random_state random_state: int, RandomState instance or None, default=NoneControls the randomness of the estimator. The features are alwaysrandomly permuted at each split, even if ``splitter`` is set to``""best""``. When ``max_features < n_features``, the algorithm willselect ``max_features`` at random at each split before finding the bestsplit among them. But the best found split may vary across differentruns, even if ``max_features=n_features``. That is the case, if theimprovement of the criterion is identical for several splits and onesplit has to be selected at random. To obtain a deterministic behaviourduring fitting, ``random_state`` has to be fixed to an integer.See :term:`Glossary <random_state>` for details.",None
,"max_leaf_nod

## Hyperparameter Tuning

Trong bài toán dự đoán `sales_revenue`, Grid Search được kết hợp với Cross Validation để tìm ra bộ hyperparameters phù hợp cho Decision Tree Regressor. Tập training có khoảng `78,789` mẫu và mô hình sử dụng các đặc trưng chính sau khi tiền xử lý, nên việc kiểm soát độ phức tạp của cây là cần thiết để hạn chế overfitting.

Trước khi thực hiện Grid Search, `CustomKFold` được sử dụng với `n_splits = 5`. Với 5 folds, mỗi lần validation vẫn có khoảng hơn `15,000` mẫu, đủ lớn để đánh giá mô hình ổn định. Đồng thời, số fold này không làm quá trình tuning quá nặng và cũng hạn chế việc đánh giá bị dao động quá nhiều, vì Decision Tree vốn là mô hình dễ có phương sai cao (`high variance`) khi dữ liệu thay đổi.

Parameter grid được sử dụng:

```python
dtr_grid = {
    'max_depth': [3, 5, 8, 12],
    'min_samples_split': [100, 500, 1000],
    'min_samples_leaf': [50, 100, 500]
}
```

Các tham số này được chọn để kiểm soát độ phức tạp của cây:

- `max_depth`: giới hạn độ sâu tối đa của cây. Nếu cây quá sâu, mô hình dễ học quá sát dữ liệu training.
- `min_samples_split`: số mẫu tối thiểu để một node tiếp tục được chia. Giá trị lớn hơn giúp hạn chế các nhánh quá nhỏ.
- `min_samples_leaf`: số mẫu tối thiểu tại mỗi leaf node. Tham số này giúp dự đoán ổn định hơn và giảm overfitting.

Metric chính dùng để chọn best estimator là `R²`, vì metric này cho biết mô hình giải thích được bao nhiêu phần biến thiên của target. Trong giai đoạn tuning, target hiện đang ở dạng logarithm, nên các metric như `R²` và `neg_RMSE` cũng được hiểu trên thang log của `sales_revenue`. Ngoài ra, `neg_rmse`, `neg_mae` và `neg_mse` có thể được dùng để phân tích thêm mức độ sai số. Với các metric lỗi dạng âm, giá trị càng gần `0` thì sai số càng nhỏ.

Bộ hyperparameters tốt nhất thu được sau khi Grid Search:

```python
{
    'max_depth': 8,
    'min_samples_split': 100,
    'min_samples_leaf': 500
}
```

Kết quả hiện tại đạt khoảng:

```text
R² ≈ 0.69
neg_RMSE ≈ -1.1233
```

Kết quả `R² ≈ 0.69` cho thấy mô hình giải thích được khoảng `69%` sự biến thiên của `log1p(sales_revenue)`. Đây là kết quả khá tốt đối với một mô hình Decision Tree đơn lẻ, đặc biệt vì Decision Tree chỉ chia dữ liệu theo các ngưỡng rời rạc và không biểu diễn trực tiếp các quan hệ mượt hoặc phức tạp như một số mô hình mạnh hơn.

Giá trị `neg_RMSE ≈ -1.1233` nghĩa là RMSE trên thang log khoảng `1.1233`. Vì target đã được log-transform, sai số này không nên hiểu trực tiếp là sai lệch `1.1233` đơn vị doanh thu. Nếu muốn diễn giải sai số theo đơn vị `sales_revenue` ban đầu, cần inverse transform cả `y_test` và `y_pred` bằng `np.expm1()` trước khi tính MAE, RMSE hoặc MSE.

Việc `min_samples_leaf = 500` cho thấy mô hình hoạt động tốt hơn khi mỗi leaf node đại diện cho một nhóm mẫu đủ lớn. Điều này giúp giảm overfitting và làm dự đoán ổn định hơn. Nhìn chung, Decision Tree Regressor là một baseline hợp lý, dễ giải thích và cho kết quả tương đối tốt sau khi tuning. Tuy nhiên, để cải thiện hiệu năng, các mô hình ensemble như Random Forest hoặc Gradient Boosting nên được thử nghiệm tiếp theo, vì chúng có thể giảm variance và khai thác quan hệ phi tuyến tốt hơn.

In [75]:
dtr_grid = {'max_depth': [3,5,8,12],
            'min_samples_split': [100,500,1000],
            'min_samples_leaf': [50,100,500]}

In [77]:
cv = CustomKFold(n_splits=5, shuffle=True, random_state=42)
scratch_dtr = DTR()

dtr_grid_search = CustomGridSearchCV(estimator=scratch_dtr,param_grid = dtr_grid,cv=cv,scoring = 'r2')
dtr_grid_search.fit(X_train, y_train)


Bắt đầu GridSearchCV: 36 tổ hợp tham số, 5 folds.
[1/36] Params: {'max_depth': 3, 'min_samples_split': 100, 'min_samples_leaf': 50} --> r2: 0.5379
[2/36] Params: {'max_depth': 3, 'min_samples_split': 100, 'min_samples_leaf': 100} --> r2: 0.5379
[3/36] Params: {'max_depth': 3, 'min_samples_split': 100, 'min_samples_leaf': 500} --> r2: 0.5380
[4/36] Params: {'max_depth': 3, 'min_samples_split': 500, 'min_samples_leaf': 50} --> r2: 0.5379
[5/36] Params: {'max_depth': 3, 'min_samples_split': 500, 'min_samples_leaf': 100} --> r2: 0.5379
[6/36] Params: {'max_depth': 3, 'min_samples_split': 500, 'min_samples_leaf': 500} --> r2: 0.5380
[7/36] Params: {'max_depth': 3, 'min_samples_split': 1000, 'min_samples_leaf': 50} --> r2: 0.5379
[8/36] Params: {'max_depth': 3, 'min_samples_split': 1000, 'min_samples_leaf': 100} --> r2: 0.5379
[9/36] Params: {'max_depth': 3, 'min_samples_split': 1000, 'min_samples_leaf': 500} --> r2: 0.5380
[10/36] Params: {'max_depth': 5, 'min_samples_split': 100, 'min_samp

## So sánh DTR không dùng thư viện và DTR dùng `sklearn`


In [ ]:
from sklearn.metrics import r2_score, mean_absolute_error, mean_squared_error

best_params = dtr_grid_search.best_params_

# Mô hình DTR from scratch đã được train lại trên toàn bộ training set trong GridSearch
scratch_dtr = dtr_grid_search.best_estimator_

# Mô hình DTR của sklearn dùng cùng bộ hyperparameters để so sánh công bằng hơn
sklearn_dtr = sklearn.tree.DecisionTreeRegressor(**best_params, random_state=42)
sklearn_dtr.fit(X_train, y_train)

scratch_pred = scratch_dtr.predict(X_test)
sklearn_pred = sklearn_dtr.predict(X_test)

def calculate_regression_metrics(y_true, y_pred):
    mse = mean_squared_error(y_true, y_pred)
    return {
        'R2': r2_score(y_true, y_pred),
        'MAE': mean_absolute_error(y_true, y_pred),
        'RMSE': np.sqrt(mse),
        'MSE': mse,
    }

comparison_df = pd.DataFrame([
    {
        'Model': 'Decision Tree Regressor - Không dùng thư viện',
        **calculate_regression_metrics(y_test.to_numpy(), scratch_pred),
    },
    {
        'Model': 'Decision Tree Regressor - Dùng `sklearn.tree`',
        **calculate_regression_metrics(y_test, sklearn_pred),
    },
])


display(comparison_df)


,Model,R2,MAE,RMSE,MSE
0,Decision Tree Regressor - Không dùng thư viện,0.488541,1777.355322,3061.452416,9.372491e+06
1,Decision Tree Regressor - Dùng `sklearn.tree`,0.488541,1777.355322,3061.452416,9.372491e+06


## Kết luận

### Hướng triển khai Decision Tree Regressor:
- From Scratch: thuật toán xây cây dựa trên việc duyệt qua các feature, thử các threshold có thể chia, tính weighted MSE cho từng cách chia và chọn split sao cho `MSE` là tối thiểu. Cách triển khai này giúp hiểu rõ hơn cơ chế hoạt động bên trong của Decision Tree, đặc biệt là cách mô hình chọn feature, threshold và tạo leaf node.
- Sử dụng `DecisionTreeRegressor` có sẵn từ `sklearn.tree`.


### So sánh hiệu năng giữa hai cách triển khai


Khi so sánh với `DecisionTreeRegressor` của `sklearn`, hai mô hình cho kết quả gần như giống hệt nhau trên tập test. Các metric như `R²`, `MAE`, `RMSE` và `MSE` gần như trùng nhau, trong khi sai khác lớn nhất giữa các giá trị dự đoán chỉ ở mức rất nhỏ do sai số tính toán số thực. Điều này cho thấy phiên bản Decision Tree Regressor tự triển khai đã mô phỏng khá sát logic cơ bản của thư viện `sklearn`.


### Hiệu năng thực sự của mô hình 


Tuy nhiên, kết quả `R²` chỉ đạt khoảng `0.48`, cho thấy mô hình chỉ giải thích được khoảng 48% sự biến thiên của `sales_revenue`. Đây là một mức kết quả còn hạn chế. N

**Nguyên nhân**: Có thể đến từ bản chất của Decision Tree, mô hình chia dữ liệu theo các ngưỡng rời rạc nên khó biểu diễn tốt các quan hệ phức tạp hoặc liên tục giữa các đặc trưng và doanh thu. Ngoài ra, Decision Tree cũng là mô hình có xu hướng *High Variance*, tức là dễ bị ảnh hưởng bởi thay đổi nhỏ trong dữ liệu nếu không được giới hạn `max_depth` hoặc `min_samples_split`.


### Hướng giải quyết tiếp theo


Nhìn chung, Decision Tree Regressor phù hợp để làm baseline model vì dễ hiểu, dễ giải thích và giúp quan sát được cách dữ liệu được chia theo từng feature. Tuy nhiên, để cải thiện hiệu năng dự đoán, các mô hình ensemble như Random Forest Regressor hoặc Gradient Boosting Regressor nên được thử nghiệm tiếp theo. Random Forest có thể giảm variance bằng cách kết hợp nhiều cây quyết định, trong khi Gradient Boosting có thể cải thiện dần sai số qua nhiều mô hình yếu liên tiếp.